# BG-forecasting — training-schedule sensitivity on Colab

Answers reviewer R3 #3: both regimes inherit Cui et al.'s schedule without
tuning, so part of the measured transfer benefit could be an artefact of
stopping the from-scratch baseline too early rather than a property of transfer.

The paper currently states the gap and leaves it untested
(§III-B "Configuration" and §IV-E). This notebook closes it, or reports that it
does not close.

### Why it matters more now than when R3 wrote it

The measured benefit fell from 4-8% to **1.18-3.64%** — 0.17-0.58 mg/dL. A
baseline halted a few epochs early can plausibly account for a difference that
size. Declaring the confound untested was defensible at the old effect size;
at this one it invites the reviewer to conclude the headline is unidentified.

### The three arms

| Arm | Regime run | RL schedule | Purpose |
|---|---|---|---|
| `arm0_patience5` | both | 50 epochs, patience 5 | the paper's schedule, replicated on this device; supplies the shared TL reference |
| `armA_patience15` | regular | 50 epochs, patience 15 | was patience 5 cutting RL short? |
| `armB_fixed20` | regular | 20 epochs, patience 20 (never fires) | RL given TL's total epoch budget |

**One correction to `TRAINING_SCHEDULE_SENSITIVITY.md`.** That document assumes
TL is unchanged by the patience edits. It is not:
`benchmark/experiments/configured.py` passes the single
`training.early_stopping_patience` to *every* `fit` call, so raising it also
removes early stopping from TL's 10-epoch pre-training and 10-epoch
fine-tuning stages. Running all three arms as `mode: both` would therefore vary
TL and RL at the same time and answer a different question. Arms A and B here
run `mode: regular` and are paired against arm 0's TL, which holds TL fixed by
construction and costs one TL training instead of three.

Everything else — cohort, seeds, features, window, horizon, architecture — comes
from `configs/full_gru_30min.yaml` unchanged.

### Read before running

**GRU at 30 minutes**, the article's headline cell, where the effect is clearly
non-zero and has room to move. Add 60 min afterwards if the result is
borderline.

**Budget ~1.5-2 h on a T4** for all three arms (arm 0 is the expensive one:
it trains both regimes). Each arm is a single `benchmark.cli run`; a disconnect
mid-arm means re-running that arm, so run them one cell at a time.

**Device.** `DEVICE_CFG = None` inherits whatever
`configs/full_gru_30min.yaml` carries, which is what the published cell was
trained with — currently `cuda`. GPU runs are seeded but not bitwise
reproducible, so arm 0 will land near Table I rather than exactly on it. That is
what arm 0 is for: all three rows are then measured on the same device in the
same session, so the comparison between them is internally consistent even
though none of them reproduces the published numbers to the last decimal. Set
`"cpu"` to trade roughly 3x the wall clock for bitwise reproducibility.

## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"    # results are mirrored here
REPO_DIR      = "/content/BG-forecasting"

SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

SEEDS      = [41, 42, 43]
DEVICE_CFG = None          # None inherits the base config's device; "cpu"/"cuda" overrides it
# ---------------------------------------------------------------------------

import os, pathlib
SENS_DRIVE = f"{DRIVE_RESULTS}/sensitivity"
for d in (SENS_DRIVE, f"{DRIVE_RESULTS}/logs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready. Arm outputs mirror to", SENS_DRIVE)

## 2 · Get the code

In [ ]:
import shutil, subprocess, sys, pathlib

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("WARNING: git pull --ff-only failed, so this checkout may be stale.\n"
                  + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

BASE_CONFIG = pathlib.Path(REPO_DIR) / "configs" / "full_gru_30min.yaml"
if not BASE_CONFIG.is_file():
    raise SystemExit(f"{BASE_CONFIG} missing; the arms are derived from it.")
print("Base config:", BASE_CONFIG)

## 3 · Stage the OhioT1DM data

In [ ]:
import shutil, pathlib

COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found. Upload your OhioT1DM copy there first.")

copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if target.is_file() or not source.is_file():
                continue
            shutil.copy2(source, target)
            copied += 1
print(f"{copied} file(s) copied from Drive.")

missing = []
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            if not (dst / release / mode / name).is_file():
                where = "absent from Drive too" if not (src / release / mode / name).is_file() else "copy failed"
                missing.append(f"{release}/{mode}/{name} ({where})")
if missing:
    raise SystemExit("Missing data files:\n  " + "\n  ".join(missing))
print("All 24 XML files staged.")

## 4 · Write the three arm configs

Derived from `configs/full_gru_30min.yaml`, changing only the experiment name,
the training mode, the RL epoch/patience schedule, the device, and the two
output switches that produce figures and prediction CSVs this check does not
read. The F=4 input flags are asserted rather than assumed: a stale checkout
would otherwise silently run a glucose-only sensitivity check against a
four-feature headline.

In [ ]:
import yaml, copy, pathlib

base = yaml.safe_load(BASE_CONFIG.read_text())

pre, train = base["preprocessing"], base["training"]
assert pre["unimodal"] is False, "base config is not the F=4 publication config"
assert pre["include_feature_engineering"] is False, "feature engineering would push F=6"
assert pre["prediction_horizon"] == 6 and pre["window_size"] == 12, "not the 30-min/1-h-history cell"
assert train["epochs"] == 50 and train["early_stopping_patience"] == 5, \
    "base config no longer carries the Cui et al. schedule this check is testing"
assert train["pretrain_epochs"] == 10 and train["finetune_epochs"] == 10
assert sorted(train["seeds"]) == sorted(SEEDS), "seed sets differ from the published run"

ARMS = {
    "sens_gru_30min_arm0_patience5": dict(
        mode="both", epochs=50, early_stopping_patience=5,
        note="paper schedule, replicated on this device; supplies TL and the RL baseline"),
    "sens_gru_30min_armA_patience15": dict(
        mode="regular", epochs=50, early_stopping_patience=15,
        note="R3 #3 directly: was patience 5 cutting RL short?"),
    "sens_gru_30min_armB_fixed20": dict(
        mode="regular", epochs=20, early_stopping_patience=20,
        note="RL matched to TL's 10+10 total epoch budget; patience never fires"),
}

written = {}
for name, arm in ARMS.items():
    cfg = copy.deepcopy(base)
    cfg["experiment"]["name"] = name
    cfg["experiment"]["description"] = f"schedule sensitivity (R3 #3): {arm['note']}"
    cfg["training"]["mode"] = arm["mode"]
    cfg["training"]["epochs"] = arm["epochs"]
    cfg["training"]["early_stopping_patience"] = arm["early_stopping_patience"]
    if DEVICE_CFG is not None:
        cfg["training"]["device"] = DEVICE_CFG
    # Not read by this check; both cost time and disk.
    cfg["output"]["save_predictions"] = False
    cfg["output"]["generate_plots"] = False
    path = pathlib.Path(REPO_DIR) / "configs" / f"{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    written[name] = path
    print(f"{name:32s} mode={arm['mode']:8s} epochs={arm['epochs']:3d} "
          f"patience={arm['early_stopping_patience']:3d}  -> {path.name}")

print("\ndevice:", cfg["training"]["device"],
      "(inherited)" if DEVICE_CFG is None else "(overridden by DEVICE_CFG)")
print("Everything else is inherited from", BASE_CONFIG.name)

## 5 · Run the arms

One cell per arm so a disconnect costs one arm, not three. Completed arms are
detected by name and skipped, so rerunning this cell is safe.

If an arm dies part-way, its parent directory has no `aggregate_metrics.json`
and will be ignored. You can either rerun the arm, or resume the individual seed
that failed:

```
python -m benchmark.cli resume --run-dir results/experiments/<parent>/regular/seed_42
```

In [ ]:
import subprocess, sys, os, pathlib, shutil, time, collections, yaml

EXP_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

# A fresh Colab VM has no local results. Restore arms finished in an earlier
# session from Drive first, or they get retrained.
restored = 0
for saved in sorted(pathlib.Path(SENS_DRIVE).glob("experiment_*")):
    if not (EXP_DIR / saved.name).exists():
        shutil.copytree(saved, EXP_DIR / saved.name)
        restored += 1
print(f"Restored {restored} arm run(s) from Drive.\n")


def find_arm(name):
    """Newest completed parent run for this experiment name, or None."""
    done = []
    for resolved in EXP_DIR.glob("*/resolved_config.yaml"):
        if not (resolved.parent / "aggregate_metrics.json").is_file():
            continue
        try:
            cfg = yaml.safe_load(resolved.read_text())
        except Exception:
            continue
        if cfg.get("experiment", {}).get("name") == name:
            done.append(resolved.parent)
    return max(done, key=lambda p: p.stat().st_mtime) if done else None


def mirror(parent):
    target = pathlib.Path(SENS_DRIVE) / parent.name
    if not target.exists():
        shutil.copytree(parent, target)
    return target


KEEP = ("Patient", "[INFO]", "Early stopping", "Completed", "Parent experiment",
        "ERROR", "Traceback", "Error")


def run_arm(name):
    existing = find_arm(name)
    if existing is not None:
        print(f"{name}: already complete at {existing.name} — skipping")
        return existing
    log_path = pathlib.Path(DRIVE_RESULTS) / "logs" / f"{name}.log"
    print(f"{name}: starting (log -> {log_path})")
    t0 = time.time()
    tail = collections.deque(maxlen=40)
    with open(log_path, "w") as log:
        proc = subprocess.Popen([sys.executable, "-m", "benchmark.cli", "run",
                                 "--config", str(written[name])],
                                cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            tail.append(line.rstrip())
            if any(k in line for k in KEEP):
                print("   ", line.rstrip(), flush=True)
        rc = proc.wait()
    if rc != 0:
        print(f"    --- last {len(tail)} line(s) ---")
        for line in tail:
            print("    " + line)
        raise SystemExit(f"{name} failed (exit {rc}); full log at {log_path}")
    parent = find_arm(name)
    if parent is None:
        raise SystemExit(f"{name} reported success but wrote no aggregate_metrics.json")
    print(f"{name}: done in {(time.time()-t0)/60:.1f} min -> {mirror(parent)}")
    return parent


arm_dirs = {}
for name in ARMS:                      # arm 0 first: it trains both regimes
    arm_dirs[name] = run_arm(name)
print("\nArms:", {k: v.name for k, v in arm_dirs.items()})

## 6 · The three-row table

Built from each arm's per-seed `metrics.json`, using the same estimator the
paper uses: patients are the inferential unit, each patient's MAE is averaged
over seeds before testing, and the 95% intervals come from 20,000 resamples that
draw patients and seeds jointly, with the **same draws applied to every row** so
the rows are comparable and the `change vs arm 0` column is a paired quantity.

`benefit` is RL − TL at the advertised horizon. TL is arm 0's throughout.

In [ ]:
import json, pathlib, numpy as np, pandas as pd
from scipy.stats import wilcoxon

REPLICATES = 20000
RESAMPLING_SEED = 42


def mae_matrix(parent, mode):
    """(n_patients, n_seeds) horizon-step MAE, patients in a fixed order."""
    per_seed = {}
    for seed in SEEDS:
        path = parent / mode / f"seed_{seed}" / "metrics.json"
        if not path.is_file():
            raise SystemExit(f"Missing {path}")
        per_seed[seed] = {int(k): float(v["mae"]) for k, v in json.loads(path.read_text()).items()}
    patients = sorted(set.intersection(*(set(v) for v in per_seed.values())))
    return patients, np.array([[per_seed[s][p] for s in SEEDS] for p in patients])


patients, tl = mae_matrix(arm_dirs["sens_gru_30min_arm0_patience5"], "transfer")
rl = {}
for name, parent in arm_dirs.items():
    arm_patients, matrix = mae_matrix(parent, "regular")
    if arm_patients != patients:
        raise SystemExit(f"{name} has a different patient set: {arm_patients}")
    rl[name] = matrix
print(f"{len(patients)} patients x {len(SEEDS)} seeds; TL from arm 0.\n")

rng = np.random.default_rng(RESAMPLING_SEED)
n, k = len(patients), len(SEEDS)
pi = rng.integers(0, n, size=(REPLICATES, n))
si = rng.integers(0, k, size=(REPLICATES, k))


def resample(matrix):
    return matrix[pi[:, :, None], si[:, None, :]].mean(axis=(1, 2))


reference = resample(rl["sens_gru_30min_arm0_patience5"] - tl)
rows = []
for name, arm in ARMS.items():
    diff = rl[name] - tl
    drawn_diff, drawn_rl = resample(diff), resample(rl[name])
    percent = 100 * drawn_diff / drawn_rl
    per_patient = diff.mean(axis=1)
    change = drawn_diff - reference
    rows.append(dict(
        arm=name.replace("sens_gru_30min_", ""),
        rl_schedule=f"{arm['epochs']}ep/pat{arm['early_stopping_patience']}",
        rl_mae=rl[name].mean(), tl_mae=tl.mean(),
        benefit=diff.mean(),
        ci_low=np.quantile(drawn_diff, .025), ci_high=np.quantile(drawn_diff, .975),
        benefit_pct=100 * diff.mean() / rl[name].mean(),
        pct_low=np.quantile(percent, .025), pct_high=np.quantile(percent, .975),
        improved=int((per_patient > 0).sum()), n_patients=n,
        wilcoxon_p=float(wilcoxon(per_patient).pvalue),
        change_vs_arm0=change.mean(),
        change_low=np.quantile(change, .025), change_high=np.quantile(change, .975)))

table = pd.DataFrame(rows)
out = pathlib.Path(REPO_DIR) / "results" / "sensitivity_summary.csv"
out.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(out, index=False)
import shutil as _sh; _sh.copy2(out, pathlib.Path(SENS_DRIVE) / out.name)

pd.set_option("display.width", 200)
print(table.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print("\nPublished GRU-30 (CPU, article Table I): TL 12.82, RL 13.22, "
      "benefit 0.397 [0.253, 0.536], 3.0%")
print("Saved:", out)

## 7 · Did patience ever bind?

The cheapest form of the answer. If RL under patience 5 already stops at about
the same epoch as RL under patience 15, then the schedule was never binding and
R3's objection is answered before the effect sizes are compared.

`epochs_completed` is recorded per patient per seed by the runner. For transfer
it is the sum of the pre-training and fine-tuning stages, so arm 0's transfer
row is reported split as well — that is where a patience change would have
altered TL, and it is the reason arms A and B run `mode: regular` only.

In [ ]:
import json, numpy as np, pandas as pd, pathlib

rows = []
for name, parent in arm_dirs.items():
    cap = ARMS[name]["epochs"]
    for mode in ("regular", "transfer"):
        if not (parent / mode).is_dir():
            continue
        epochs, pretrain, finetune = [], [], []
        for seed in SEEDS:
            metrics = json.loads((parent / mode / f"seed_{seed}" / "metrics.json").read_text())
            for entry in metrics.values():
                history = entry["training_history"]
                epochs.append(history["epochs_completed"])
                if "pretrain_history" in history:
                    pretrain.append(history["pretrain_history"]["epochs_completed"])
                    finetune.append(history["finetune_history"]["epochs_completed"])
        epochs = np.array(epochs)
        row = dict(arm=name.replace("sens_gru_30min_", ""), mode=mode,
                   patience=ARMS[name]["early_stopping_patience"],
                   cap=cap if mode == "regular" else 20,
                   mean_epochs=epochs.mean(), median_epochs=float(np.median(epochs)),
                   min_epochs=int(epochs.min()), max_epochs=int(epochs.max()),
                   n_fits=len(epochs))
        if mode == "regular":
            row["pct_hitting_cap"] = 100 * float((epochs >= cap).mean())
        else:
            row["pct_hitting_cap"] = np.nan
            row["mean_pretrain_epochs"] = float(np.mean(pretrain))
            row["mean_finetune_epochs"] = float(np.mean(finetune))
        rows.append(row)

epoch_table = pd.DataFrame(rows)
out = pathlib.Path(REPO_DIR) / "results" / "sensitivity_epochs.csv"
epoch_table.to_csv(out, index=False)
import shutil as _sh; _sh.copy2(out, pathlib.Path(SENS_DRIVE) / out.name)
print(epoch_table.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
print("\nSaved:", out)

## 8 · Optional cross-check with the repository's own estimator

`RUN/experiments/run_shift_analysis.py --compare-transfer-benefit` is what
produced the published `transfer_benefit_effect_bootstrap.csv`. It needs one
parent holding both regimes, so it can only read arm 0 — which is exactly the
row that should match the published pipeline. Agreement here means section 6's
bootstrap is not quietly doing something different from the paper's.

Skipped automatically if `dataset_signal_features.csv` has not been generated
(section 10 of `run_on_colab.ipynb`, or `bash RUN/run_dataset_analysis.sh signal`).

In [ ]:
import subprocess, sys, pathlib, pandas as pd

features = pathlib.Path(REPO_DIR) / "results/analysis/dataset/dataset_signal_features.csv"
drive_features = pathlib.Path(DRIVE_RESULTS) / "analysis/dataset/dataset_signal_features.csv"
if not features.is_file() and drive_features.is_file():
    features.parent.mkdir(parents=True, exist_ok=True)
    import shutil as _sh; _sh.copy2(drive_features, features)

if not features.is_file():
    print("dataset_signal_features.csv not available — skipping the cross-check.\n"
          "Section 6 stands on its own; this cell only corroborates it.")
else:
    parent = arm_dirs["sens_gru_30min_arm0_patience5"]
    outdir = pathlib.Path(REPO_DIR) / "results/analysis/sensitivity_arm0"
    proc = subprocess.run(
        [sys.executable, "RUN/experiments/run_shift_analysis.py",
         "--configured-aggregate", str(parent / "aggregate_metrics.json"),
         "--mode", "transfer", "--compare-transfer-benefit",
         "--model", "GRU", "--data-root", "data",
         "--bootstrap-count", "20000", "--resampling-seed", "42",
         "--output-dir", str(outdir)],
        cwd=REPO_DIR, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print("FAILED:\n", proc.stderr[-3000:])
    else:
        bootstrap = outdir / "transfer_benefit_effect_bootstrap.csv"
        if bootstrap.is_file():
            print(pd.read_csv(bootstrap).to_string(index=False))
        import shutil as _sh
        _sh.copytree(outdir, pathlib.Path(SENS_DRIVE) / outdir.name, dirs_exist_ok=True)

## 9 · How to read the result

Compare the `change_vs_arm0` column and its interval — that is the paired
quantity, and reading the raw benefits across rows understates the precision.

**The interval for both arms contains zero, and the stopping epochs barely
move.** The objection is closed. Replace the "remains untested" sentences in
§III-B and §IV-E with one sentence and the three-row table: the RL schedule was
checked at patience 15 and at a matched 20-epoch budget, and the transfer
benefit is unchanged. This is the outcome that costs the paper nothing and
removes a reviewer's open question.

**The benefit shrinks materially under a longer-running RL baseline.** Then part
of the published gap came from stopping RL early, and the paper has to say so.
Uncomfortable, but it is the paper's own argument: the whole point of the
protocol is that measured transfer benefits shrink under stricter measurement,
and finding it yourself is far better than a reviewer finding it. The
Cui et al. reconciliation still holds at a smaller number, and §IV-E already
frames the six-week setting as least favourable to transfer.

**The benefit grows.** Report it plainly too; patience 5 was, if anything,
generous to RL.

**Either way**, one sentence goes into §IV-E recording that the schedule was
inherited from Cui et al. and checked at one architecture and one horizon, not
tuned per architecture. And if the result is borderline, run the same three arms
at 60 minutes, where RNN-60 is the cell that fails BH correction.

### Files produced

```
results/sensitivity_summary.csv    the three-row table (mirrored to Drive)
results/sensitivity_epochs.csv     stopping-epoch diagnostics
results/analysis/sensitivity_arm0/ optional cross-check outputs
<DRIVE_RESULTS>/sensitivity/       every arm's full experiment directory
<DRIVE_RESULTS>/logs/sens_*.log    per-arm training logs
```